In [ ]:
import scipy.signal.signaltools

def _centered(arr, newsize):
    # Return the center newsize portion of the array.
    newsize = np.asarray(newsize)
    currsize = np.array(arr.shape)
    startind = (currsize - newsize) // 2
    endind = startind + newsize
    myslice = [slice(startind[k], endind[k]) for k in range(len(endind))]
    return arr[tuple(myslice)]

scipy.signal.signaltools._centered = _centered
import seaborn as sns
import eventstudy as es
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import pandas_datareader as web

In [ ]:
df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/dfMinute.pkl")

In [ ]:
df.columns.values

In [ ]:
# df = df.loc[df["Type"]=="MERGE"]

In [ ]:
df = df[["VIX return","Sentiment return"]]
df.dropna(inplace=True)
df = df[(df.T != 0).any()]
df = df.loc[df["Sentiment return"]!=0]
df = df.loc[df["VIX return"]!=0]
df_dates = df
df_dates["Time"] = df_dates.index
df_dates['Time']=df_dates['Time'].dt.strftime('%Y-%m-%d %H:%M:%S')
df_dates.index = df_dates['Time']

In [ ]:
df_dates

In [ ]:
events_db = []
for date, row in df_dates.T.iteritems():
    print(date)
    new_row = {'event_date': np.datetime64(row["Time"]),'market_ticker':'Sentiment return','security_ticker': 'VIX return'}#row["Company"]}
    events_db.append(new_row)

In [ ]:
events_db

In [ ]:
returns = df_dates[["Sentiment return","VIX return"]]
returns.index=df_dates.Time

In [ ]:
len(df_dates)

In [ ]:
len(events_db)

In [ ]:
returns.index.names = ['date']

In [ ]:
returns.to_csv("../CSV/returns_list.csv")
es.Single.import_returns("../CSV/returns_list.csv")

In [ ]:
# es.Single.import_FamaFrench("./CSV/FamaFrenchSPY.csv")

In [ ]:
tickers={"Sentiment return"}

In [ ]:
events=events_db

In [ ]:
# tickers = ['SPY']
releases = dict()

events = [event for event in events_db if event['security_ticker']=="VIX return"]
releases["VIX return"] = es.Multiple.from_list(
    events,
    es.Single.market_model,#FamaFrench_3factor,
    estimation_size=10000,
    event_window= (-15,+15)
)
print("##########################")

In [ ]:
events

In [ ]:
len(events)

In [ ]:
releases.values

In [ ]:
ticker

In [ ]:
for ticker, event in releases.items():
    event.plot()
    print(ticker)
    # plt.figure(figsize=(20, 20))
    plt.xlabel('Event Day')
    plt.title('Event Study - Minute Frequency\nSentiment Return and VIX Return')
    plt.show()

In [ ]:
releases.items()

In [ ]:
import json
with open('convert.txt', 'w') as convert_file:
     convert_file.write(str(releases.items()))

In [ ]:
with open('convert.txt', 'a') as convert_file:
    ev = convert_file